In [5]:
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_community.document_loaders import JSONLoader

In [ ]:
from dotenv import load_dotenv
import os
from googleapiclient.discovery import build

load_dotenv()
API_KEY = os.getenv("api_key")
PLAYLIST_ID = "PLKnIA16_Rmvbr7zKYQuBfsVkjoLcJgxHH"

youtube = build("youtube","v3",developerKey=API_KEY)

video_ids = []
next_page = None

while True:
    request = youtube.playlistItems().list(
        part="contentDetails",
        playlistId=PLAYLIST_ID,
        maxResults=50,
        pageToken=next_page
    )

    response = request.execute()

    for item in response["items"]:
        video_ids.append(item["contentDetails"]["videoId"])

    next_page = response.get("nextPageToken")

    if not next_page:
        break

print(len(video_ids))

134


In [3]:
selected_videos = video_ids[0:33]

In [ ]:
import yt_dlp
import whisper
import os
import torch


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


model = whisper.load_model("medium", device=device)

def get_transcript_via_whisper(video_id):
    url = f"https://www.youtube.com/watch?v={video_id}"
    audio_filename = f"{video_id}.m4a"
    
    # 1. Download only the audio
    ydl_opts = {
        'format': 'm4a/bestaudio/best',
        'outtmpl': audio_filename,
        'quiet': True,
        'noplaylist': True,
    }
    
    try:
        print(f" Downloading audio for {video_id}...")
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])
        
        # 2. Transcribe using Whisper
        print(f" Transcribing {video_id} using {device}...")
        result = model.transcribe(audio_filename)
        
        # 3. Cleanup audio file
        os.remove(audio_filename)
        
        return result['text'].strip()
        
    except Exception as e:
        print(f" Error: {e}")
        if os.path.exists(audio_filename):
            os.remove(audio_filename)
        return None



all_transcripts = {}

for v_id in selected_videos:
    text = get_transcript_via_whisper(v_id)
    if text:
        all_transcripts[v_id] = text
        print(f"✅ Completed: {v_id} ({len(text)} characters)")

# Save to file
import json
with open("transcripts.json", "w", encoding="utf-8") as f:
    json.dump(all_transcripts, f, indent=4)

In [ ]:
loader = JSONLoader(
    file_path="transcripts.json",
    jq_schema=".[]",
    text_content=True
)

docs = list(loader.lazy_load())

In [ ]:
docs

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter,Language
splitter=RecursiveCharacterTextSplitter.from_language(language=Language.MARKDOWN,
                                        chunk_size=800,chunk_overlap=150)

chunks=splitter.split_documents(docs)

In [14]:
len(chunks)

45

In [ ]:
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import Chroma




# 1. Initialize your model
embedding_model = OllamaEmbeddings(
    model="qwen3-embedding:latest"
)



# 3. Create the vector store using 'chunks' directly
vector_store = Chroma.from_documents(
    documents=chunks,  
    embedding=embedding_model,
    persist_directory="./new_chroma_db",
    collection_name="sample"
)

print(f"Success! {len(chunks)} transcript chunks are now embedded.")

Success! 45 transcript chunks are now embedded.


In [ ]:
vector_store.get(include=["embeddings","metadatas","documents"])

In [16]:
def retriever(question):
  Retriever = vector_store.as_retriever( search_type="similarity",
                                        search_kwargs={ "k": 4 } )
  docs=Retriever.invoke(question)
  context="\n\n".join(doc.page_content for doc in docs)
  return context

In [17]:
from langchain_ollama import ChatOllama
model=ChatOllama(model="qwen3.5:9b")

In [18]:
from langchain_core.prompts import PromptTemplate

In [19]:
prompt = PromptTemplate(
    template="""
You are a question-answering assistant.

Use ONLY the information provided in the context below.
Do NOT use prior knowledge or outside information.

If the context is empty or irrelevant to the question, reply with:
"I don't know."

If the question asks for a summary, summarize the context faithfully.

Context:
{context}

Question:
{question}

Answer:
""",
    input_variables=["context", "question"],
)

In [ ]:
from langchain_core.runnables import RunnableParallel,RunnablePassthrough
parallel_chain=RunnableParallel({"context":retriever,"question":RunnablePassthrough()})


In [ ]:
from langchain_core.output_parsers import StrOutputParser
parser=StrOutputParser()

In [24]:
chain=parallel_chain|prompt|model|parser

In [ ]:
question="can you summarize the video"
chain.invoke(question)